# Carlos OS Puro 0.10 — pré-verificação do Colab

Este notebook **não compila nem altera discos**. Ele mede a sessão atual, testa permissões em diretórios temporários, monta o Google Drive e grava um pequeno arquivo de diagnóstico para testar restauração.

1. Execute a célula de diagnóstico e conclua a autorização do Google Drive na própria tela do Colab.
2. Para testar a retomada após queda de sessão, reinicie/conecte outro runtime e execute a célula de restauração.
3. Confira CPU, RAM, discos, permissões e a indicação de `/dev/kvm`. O tempo restante e a permissão do seu plano para compilação longa precisam ser conferidos no próprio Colab; o runtime não fornece uma garantia de duração.

A sessão deve estar conectada antes de executar. Nenhuma credencial é solicitada pelo código.

In [ ]:
import os, sys, json, time, stat, shutil, platform, tempfile, subprocess
from pathlib import Path
from datetime import datetime, timezone

print("=== Carlos OS Puro 0.10 | pré-verificação, sem compilação ===")
print("Data UTC:", datetime.now(timezone.utc).isoformat(timespec="seconds"))
print("Usuário efetivo:", os.geteuid(), "(0 significa root)")
print("Python:", sys.version.split()[0], "| SO:", platform.platform())

# CPU e RAM
cpu_model = "não identificado"
try:
    for line in Path("/proc/cpuinfo").read_text(errors="replace").splitlines():
        if line.lower().startswith("model name"):
            cpu_model = line.split(":", 1)[1].strip()
            break
except OSError:
    pass
mem = {}
try:
    for line in Path("/proc/meminfo").read_text().splitlines():
        key, value = line.split(":", 1)
        if key in ("MemTotal", "MemAvailable"):
            mem[key] = int(value.split()[0]) / 1024 / 1024
except (OSError, ValueError):
    pass
print(f"CPU: {os.cpu_count()} núcleos | {cpu_model}")
print("RAM total/disponível (GiB):", {k: round(v, 2) for k, v in mem.items()})
try:
    print(subprocess.run(["lscpu"], text=True, capture_output=True, timeout=10).stdout[:5000])
except Exception as e:
    print("lscpu indisponível:", e)

# Discos e ponto de montagem do disco extra
for cmd in (["df", "-hT"], ["lsblk", "-o", "NAME,LABEL,SIZE,FSTYPE,MOUNTPOINTS"]):
    if shutil.which(cmd[0]):
        r = subprocess.run(cmd, text=True, capture_output=True, timeout=15)
        print("\n$", " ".join(cmd), "\n", r.stdout or r.stderr)
print("\nMontagem de /content:")
if shutil.which("df"):
    print(subprocess.run(["df", "-hT", "/content"], text=True, capture_output=True, timeout=10).stdout)

# GPU só como inventário; ela não acelera a compilação dos pacotes LFS.
if shutil.which("nvidia-smi"):
    r = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], text=True, capture_output=True, timeout=10)
    print("GPU (não usada para compilar):", r.stdout.strip() or r.stderr.strip())
else:
    print("GPU: nvidia-smi indisponível")

# Idade observável do runtime não é tempo restante da sessão.
try:
    uptime_s = float(Path("/proc/uptime").read_text().split()[0])
    print(f"Uptime do runtime: {uptime_s / 3600:.2f} h (não informa o tempo restante do Colab)")
except Exception as e:
    print("Uptime indisponível:", e)
print("TEMPO RESTANTE: o Colab não o expõe de forma confiável ao notebook.")
print("PLANO/REGRAS: confirme na sua conta e nas regras oficiais do Colab antes de iniciar uma compilação longa.")
print("Este notebook não inicia build nem tenta contornar limites do serviço.")

# Disponibilidade de QEMU, ferramenta de loop e /dev/kvm.
for name in ("qemu-system-x86_64", "proot", "losetup", "mount", "chroot", "unshare", "mknod", "zstd"):
    print(f"Ferramenta {name}: {shutil.which(name) or 'ausente'}")
kvm = Path("/dev/kvm")
print("/dev/kvm:", "presente" if kvm.exists() else "ausente", "| leitura/escrita:", os.access(kvm, os.R_OK | os.W_OK) if kvm.exists() else False)

# Testes de capacidade sem formatar dispositivos e usando somente arquivos temporários.
results = {}
def run_probe(label, args, timeout=10):
    try:
        r = subprocess.run(args, text=True, capture_output=True, timeout=timeout)
        detail = (r.stderr or r.stdout).strip().replace("\n", " ")[:240]
        results[label] = {"ok": r.returncode == 0, "detalhe": detail or f"código {r.returncode}"}
    except Exception as e:
        results[label] = {"ok": False, "detalhe": f"{type(e).__name__}: {e}"}

def mount_probe(kind):
    if not shutil.which("mount") or not shutil.which("umount"):
        return False, "mount/umount ausente"
    with tempfile.TemporaryDirectory(prefix="carlos-puro-mount-", dir="/content") as root:
        base = Path(root)
        if kind == "bind":
            src, dst = base / "src", base / "dst"
            src.mkdir(); dst.mkdir()
            args = ["mount", "--bind", str(src), str(dst)]
        else:
            dst = base / "proc"
            dst.mkdir()
            args = ["mount", "-t", "proc", "proc", str(dst)]
        r = subprocess.run(args, text=True, capture_output=True, timeout=10)
        if r.returncode != 0:
            return False, (r.stderr or r.stdout).strip()[:240]
        u = subprocess.run(["umount", str(dst)], text=True, capture_output=True, timeout=10)
        if u.returncode != 0:
            return False, "mount passou, mas desmontagem falhou: " + (u.stderr or u.stdout).strip()[:200]
        return True, "montou e desmontou em diretório temporário"

try:
    run_probe("chroot", ["chroot", "/", "/usr/bin/true"])
except Exception as e:
    results["chroot"] = {"ok": False, "detalhe": str(e)}
for label, kind in (("mount --bind", "bind"), ("mount -t proc", "proc")):
    try:
        ok, detail = mount_probe(kind)
        results[label] = {"ok": ok, "detalhe": detail}
    except Exception as e:
        results[label] = {"ok": False, "detalhe": f"{type(e).__name__}: {e}"}

try:
    with tempfile.TemporaryDirectory(prefix="carlos-puro-mknod-", dir="/content") as root:
        node = Path(root) / "null-probe"
        os.mknod(node, stat.S_IFCHR | 0o600, os.makedev(1, 3))
        results["mknod"] = {"ok": node.is_char_device(), "detalhe": "nó de dispositivo temporário criado"}
except Exception as e:
    results["mknod"] = {"ok": False, "detalhe": f"{type(e).__name__}: {e}"}

if shutil.which("unshare"):
    run_probe("unshare", ["unshare", "--user", "--map-root-user", "true"])
else:
    results["unshare"] = {"ok": False, "detalhe": "ferramenta ausente"}

if shutil.which("losetup"):
    try:
        with tempfile.TemporaryDirectory(prefix="carlos-puro-loop-", dir="/content") as root:
            image = Path(root) / "loop-probe.img"
            with image.open("wb") as f:
                f.truncate(8 * 1024 * 1024)
            r = subprocess.run(["losetup", "--find", "--show", str(image)], text=True, capture_output=True, timeout=10)
            if r.returncode != 0:
                results["loop"] = {"ok": False, "detalhe": (r.stderr or r.stdout).strip()[:240]}
            else:
                device = r.stdout.strip()
                d = subprocess.run(["losetup", "--detach", device], text=True, capture_output=True, timeout=10)
                results["loop"] = {"ok": d.returncode == 0, "detalhe": f"anexou/desanexou {device}" if d.returncode == 0 else (d.stderr or d.stdout).strip()[:240]}
    except Exception as e:
        results["loop"] = {"ok": False, "detalhe": f"{type(e).__name__}: {e}"}
else:
    results["loop"] = {"ok": False, "detalhe": "losetup ausente"}

print("\n=== Permissões ===")
for label, value in results.items():
    print(("OK  " if value["ok"] else "NÃO") + f" {label}: {value['detalhe']}")

# Autoriza o Google Drive na própria interface do Colab; nenhuma credencial é lida pelo código.
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    drive_root = Path("/content/drive/MyDrive")
    if not drive_root.is_dir():
        raise RuntimeError("MyDrive não apareceu após a montagem")
    project_dir = drive_root / "CarlosOS_Puro" / "diagnostico"
    project_dir.mkdir(parents=True, exist_ok=True)
    print("Drive montado:", drive_root)
    print("Espaço atual do Drive visível por df:")
    print(subprocess.run(["df", "-hT", str(drive_root)], text=True, capture_output=True, timeout=10).stdout)

    # Checkpoint pequeno, somente dentro da pasta nova do projeto. Nada é apagado.
    if not shutil.which("tar") or not shutil.which("zstd"):
        print("Checkpoint .tar.zst não testado: tar ou zstd ausente; nenhuma instalação foi feita.")
    else:
        stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
        probe = project_dir / f"retomada-probe-{stamp}.tar.zst"
        with tempfile.TemporaryDirectory(prefix="carlos-puro-probe-", dir="/content") as srcdir:
            src = Path(srcdir) / "retomada-probe"
            src.mkdir()
            (src / "verificacao.txt").write_text("Carlos OS Puro: teste de gravação no Drive; sem compilação.\n", encoding="utf-8")
            r = subprocess.run(["tar", "--zstd", "-cf", str(probe), "-C", srcdir, "retomada-probe"], text=True, capture_output=True, timeout=60)
            if r.returncode != 0:
                raise RuntimeError("Não foi possível criar checkpoint: " + (r.stderr or r.stdout).strip()[:300])
        print("Checkpoint de diagnóstico salvo:", probe)
        print("Ele é pequeno e não contém fontes, credenciais ou dados pessoais.")
        print("Execute a célula seguinte depois de reconectar outro runtime para testar a restauração após queda.")
except Exception as e:
    print("Drive/checkpoint não verificado:", f"{type(e).__name__}: {e}")


## Teste de retomada

Para simular a queda entre checkpoints, execute a primeira célula, reinicie o runtime e então execute a célula abaixo. O arquivo de probe fica em `MyDrive/CarlosOS_Puro/diagnostico/`; ele é pequeno e não será apagado pelo notebook.


In [ ]:
# Teste de restauração do checkpoint de diagnóstico salvo na célula anterior.
# Para simular a recuperação entre sessões, reinicie/conecte outro runtime antes desta célula.
import shutil, subprocess, tempfile
from pathlib import Path

if not Path("/content/drive/MyDrive").is_dir():
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
probe_dir = Path("/content/drive/MyDrive/CarlosOS_Puro/diagnostico")
archives = sorted(probe_dir.glob("retomada-probe-*.tar.zst"), key=lambda p: p.stat().st_mtime, reverse=True) if probe_dir.exists() else []
if not archives:
    raise RuntimeError("Nenhum checkpoint probe encontrado no Drive. Execute primeiro a célula de diagnóstico.")
archive = archives[0]
if not shutil.which("tar") or not shutil.which("zstd"):
    raise RuntimeError("tar ou zstd não está disponível neste runtime; não instalei dependências automaticamente.")

# Confere tamanho e lê somente o arquivo esperado para stdout; não extrai caminhos
# controlados por conteúdo externo para o sistema de arquivos.
if archive.stat().st_size > 100_000:
    raise RuntimeError("O probe é maior que o esperado; nada foi restaurado")
r = subprocess.run(["tar", "--zstd", "-xOf", str(archive), "retomada-probe/verificacao.txt"], capture_output=True, timeout=30)
if r.returncode != 0:
    raise RuntimeError("Falha ao ler o arquivo: " + r.stderr.decode("utf-8", "replace")[:300])
expected = "Carlos OS Puro: teste de gravação no Drive; sem compilação.\n".encode("utf-8")
if r.stdout != expected:
    raise RuntimeError("O conteúdo restaurado não confere")
with tempfile.TemporaryDirectory(prefix="carlos-puro-restore-", dir="/content") as destination:
    restored = Path(destination) / "verificacao.txt"
    restored.write_bytes(r.stdout)
    if restored.read_bytes() != expected:
        raise RuntimeError("A gravação temporária do conteúdo restaurado falhou")

print("RESTAURAÇÃO OK:", archive)
print("A leitura do arquivo persistente do Drive e a extração temporária funcionaram.")
print("Este foi um probe, não uma retomada de compilação; nenhum pacote foi compilado.")
